In [ ]:
!pip install folium


In [ ]:
import pandas as pd
import folium


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import pandas as pd
import numpy as np

# 1️⃣ Load deforestation points
df_def = pd.read_csv('/content/drive/MyDrive/Deforestation_Predictions.csv')
df_deforest = df_def[df_def['deforestation']==1]

# 2️⃣ Load time-series indices
df_ts = pd.read_csv('/content/drive/MyDrive/GEE_Forest_TimeSeries/TamilNadu_Forest_TimeSeries_Master.csv')

# 3️⃣ Sort and compute changes
df_ts_sorted = df_ts.sort_values(by=['latitude','longitude','year'])

# First and last year values per point
first_year = df_ts_sorted.groupby(['latitude','longitude']).first().reset_index()
last_year  = df_ts_sorted.groupby(['latitude','longitude']).last().reset_index()

df_change = pd.merge(first_year, last_year, on=['latitude','longitude'], suffixes=('_start','_end'))

# Compute delta indices
df_change['NDVI_change'] = df_change['NDVI_end'] - df_change['NDVI_start']
df_change['NBR_change']  = df_change['NBR_end']  - df_change['NBR_start']
df_change['BSI_change']  = df_change['BSI_end']  - df_change['BSI_start']
df_change['NDMI_change'] = df_change['NDMI_end'] - df_change['NDMI_start']


In [ ]:
# Function to compute thresholds
def get_thresholds(series):
    mean = series.mean()
    std = series.std()
    q1 = series.quantile(0.25)
    q3 = series.quantile(0.75)
    return {
        'mean': mean,
        'std': std,
        'q1': q1,
        'q3': q3,
        'lower_2std': mean - 2*std,
        'upper_2std': mean + 2*std
    }

# Compute thresholds for each index
ndvi_thresh = get_thresholds(df_change['NDVI_change'])
nbr_thresh  = get_thresholds(df_change['NBR_change'])
bsi_thresh  = get_thresholds(df_change['BSI_change'])
ndmi_thresh = get_thresholds(df_change['NDMI_change'])

print("NDVI thresholds:", ndvi_thresh)
print("NBR thresholds:", nbr_thresh)
print("BSI thresholds:", bsi_thresh)
print("NDMI thresholds:", ndmi_thresh)


NDVI thresholds: {'mean': np.float64(-0.16751523074070834), 'std': 0.10965343607237009, 'q1': np.float64(-0.2423575925), 'q3': np.float64(-0.08911268250000001), 'lower_2std': np.float64(-0.3868221028854485), 'upper_2std': np.float64(0.051791641404031835)}
NBR thresholds: {'mean': np.float64(-0.11094423569831657), 'std': 0.1322079900071495, 'q1': np.float64(-0.1926299775), 'q3': np.float64(-0.024758325249999998), 'lower_2std': np.float64(-0.3753602157126156), 'upper_2std': np.float64(0.15347174431598243)}
BSI thresholds: {'mean': np.float64(0.02542046754910951), 'std': 0.08549605343108485, 'q1': np.float64(-0.028495068406114946), 'q3': np.float64(0.07840941215289587), 'lower_2std': np.float64(-0.1455716393130602), 'upper_2std': np.float64(0.1964125744112792)}
NDMI thresholds: {'mean': np.float64(-0.03848384904800983), 'std': 0.09882654347987382, 'q1': np.float64(-0.0956455715), 'q3': np.float64(0.01839249055), 'lower_2std': np.float64(-0.23613693600775748), 'upper_2std': np.float64(0.15

In [ ]:
df_merged = pd.merge(df_deforest, df_change[['latitude','longitude','NDVI_change','NBR_change','BSI_change','NDMI_change']],
                     on=['latitude','longitude'], how='left')


In [ ]:
def classify_cause_adaptive(row):
    ndvi = row['NDVI_change']
    nbr  = row['NBR_change']
    bsi  = row['BSI_change']

    # Fire: large drop in NBR beyond 2 std deviations
    if nbr < nbr_thresh['lower_2std']:
        return "Fire"
    # Logging: NDVI drop below 1st quartile + BSI increase above 3rd quartile
    elif ndvi < ndvi_thresh['q1'] and bsi > bsi_thresh['q3']:
        return "Logging"
    # Agriculture: moderate NDVI drop between q1 and mean, small BSI increase
    elif ndvi < ndvi_thresh['mean'] and bsi > bsi_thresh['q1']:
        return "Agriculture"
    else:
        return "Urban/Other"

# Apply classification
df_merged['cause'] = df_merged.apply(classify_cause_adaptive, axis=1)


In [ ]:
df_merged.to_csv('/content/drive/MyDrive/Deforestation_Causes_Adaptive.csv', index=False)

# Summary
print("Deforestation Cause Summary (Adaptive Thresholds):")
print(df_merged['cause'].value_counts())


Deforestation Cause Summary (Adaptive Thresholds):
cause
Urban/Other    933
Agriculture    207
Logging         93
Fire            13
Name: count, dtype: int64


In [ ]:
import pandas as pd
import folium
from folium.plugins import MarkerCluster

# Load improved cause CSV
df = pd.read_csv('/content/drive/MyDrive/Deforestation_With_Cause_Improved.csv')

print(df['cause'].value_counts())


cause
Urban/Other    687
Fire           547
Agriculture     13
Logging          8
Name: count, dtype: int64


In [ ]:
# Tamil Nadu roughly: latitude 8–13, longitude 76–80
m = folium.Map(location=[11.0, 78.5], zoom_start=7)


In [ ]:
cause_colors = {
    'Fire': 'red',
    'Logging': 'orange',
    'Agriculture': 'yellow',
    'Urban/Other': 'gray'
}


In [ ]:
# Cluster points for better performance
marker_cluster = MarkerCluster().add_to(m)

for _, row in df.iterrows():
    color = cause_colors.get(row['cause'], 'blue')

    folium.CircleMarker(
        location=[row['latitude'], row['longitude']],
        radius=3,
        color=color,
        fill=True,
        fill_opacity=0.7,
    ).add_to(marker_cluster)


In [ ]:
m


In [ ]:
m.save('/content/drive/MyDrive/TamilNadu_Deforestation_Causes_Adaptive_Map.html')


In [ ]:
import pandas as pd
import numpy as np

# Load adaptive cause CSV
df = pd.read_csv('/content/drive/MyDrive/Deforestation_Causes_Adaptive.csv')

# Check data
print(df.head())


   latitude  longitude  deforestation  NDVI_change  NBR_change  BSI_change  \
0  8.995280  77.145071              1    -0.336893   -0.259790    0.133771   
1  9.000670  77.135189              1    -0.298443   -0.202697    0.099690   
2  9.000670  77.138783              1    -0.309123   -0.194289    0.095126   
3  9.000670  77.143274              1    -0.275817   -0.176757    0.056194   
4  9.001568  77.136986              1    -0.314450   -0.215500    0.082848   

   NDMI_change        cause  
0    -0.131280      Logging  
1    -0.088259      Logging  
2    -0.071421      Logging  
3    -0.060062  Agriculture  
4    -0.073192      Logging  


In [ ]:
# Define grid size in degrees (~5 km ≈ 0.045 degrees)
grid_size = 0.045

# Assign each point to a grid cell
df['lat_grid'] = (df['latitude'] // grid_size) * grid_size
df['lon_grid'] = (df['longitude'] // grid_size) * grid_size


In [ ]:
# Group by grid and cause
grid_summary = df.groupby(['lat_grid', 'lon_grid', 'cause']).size().reset_index(name='deforestation_count')

# Pivot table to see all causes per grid
grid_pivot = grid_summary.pivot_table(index=['lat_grid','lon_grid'], columns='cause', values='deforestation_count', fill_value=0)

# Optional: total deforestation per grid
grid_pivot['total_deforestation'] = grid_pivot.sum(axis=1)

# Sort by total deforestation descending
top_hotspots = grid_pivot.sort_values(by='total_deforestation', ascending=False).head(10)


In [ ]:
print("Top 10 Deforestation Hotspots (Grid-Based):")
print(top_hotspots)


Top 10 Deforestation Hotspots (Grid-Based):
cause              Agriculture  Fire  Logging  Urban/Other  \
lat_grid lon_grid                                            
9.945    79.020            8.0   0.0      0.0        137.0   
9.900    78.975           11.0   0.0      1.0        123.0   
         79.020            7.0   1.0      0.0        114.0   
9.945    78.975           12.0   2.0      3.0        101.0   
         79.110            6.0   1.0      2.0        107.0   
9.900    79.110            6.0   0.0      0.0        107.0   
9.945    79.065           11.0   0.0      0.0         84.0   
12.240   79.920           44.0   0.0     21.0         10.0   
9.945    79.155            3.0   0.0      4.0         60.0   
9.900    79.065           10.0   0.0      3.0         46.0   

cause              total_deforestation  
lat_grid lon_grid                       
9.945    79.020                  145.0  
9.900    78.975                  135.0  
         79.020                  122.0  
9.945 